[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/general-sense/xdflow/blob/main/examples/notebooks/evaluate.ipynb)

# Evaluate your model in one call

You have data and a scikit-learn model. This notebook shows what xdflow adds with one function call:

1. `xdflow.evaluate` cross-validates the model with each person's data on one side of every split, and reports the score per fold, per person and per trial.
2. `xdflow.check_split` audits a split you already use, without running a model.
3. `@xdflow.step` turns your own preprocessing into a pipeline step that runs once for all folds.
4. `xdflow.to_sklearn` puts an xdflow pipeline back into scikit-learn tools.

The data is the [UCI Human Activity Recognition](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones) dataset: 2.56 s windows of a waist-worn phone's accelerometer and gyroscope from 30 people, labelled with one of six activities. The first run downloads 61 MB; everything else takes about a minute on a laptop.

In [ ]:
%pip install -q "xdflow>=0.4"

## Load the data

The loader returns one `DataArray` with `trial`, `channel` and `time` dims. Every window carries its `activity`, `subject` and `segment` (a run of consecutive, overlapping windows from one person doing one activity).

In [1]:
import os
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import xdflow
from xdflow.transforms.basic_transforms import FlattenTransform

URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
ROOT = "UCI HAR Dataset"
CHANNELS = [f"{sensor}_{axis}" for sensor in ("body_acc", "body_gyro", "total_acc") for axis in "xyz"]
FS = 50  # Hz; every window is 128 samples = 2.56 s
BANDS = ((0, 1), (1, 3), (3, 6), (6, 12), (12, 26))  # Hz
FEATURES = ["mean", "std", "min", "max", "p25", "p75"] + [f"logpow_{lo}_{hi}hz" for lo, hi in BANDS]
DATA_DIR = Path(os.environ.get("XDFLOW_DATA_DIR", "~/xdflow_data")).expanduser()

In [2]:
def check_archive(archive: Path) -> Path:
    """Fail loudly unless the dataset archive has every file the loader reads."""
    expected = {f"{ROOT}/activity_labels.txt"} | {
        f"{ROOT}/{split}/{name}_{split}.txt"
        for split in ("train", "test")
        for name in ("y", "subject", *(f"Inertial Signals/{channel}" for channel in CHANNELS))
    }
    with zipfile.ZipFile(archive) as z:
        missing = expected - set(z.namelist())
    if missing:
        raise RuntimeError(f"{archive} lacks {sorted(missing)}; delete it to download again.")
    return archive


def fetch_har(data_dir: Path) -> Path:
    """Download once and return the checked inner dataset archive."""
    archive = data_dir / f"{ROOT}.zip"
    if not archive.exists():
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {URL} (61 MB) to {data_dir} ...")
        with tempfile.TemporaryDirectory(dir=data_dir) as tmp:  # only a complete, checked archive enters the cache
            outer = Path(tmp) / "download.zip"
            urllib.request.urlretrieve(URL, outer)
            with zipfile.ZipFile(outer) as z:
                z.extract(archive.name, tmp)
            os.replace(check_archive(Path(tmp) / archive.name), archive)
    return check_archive(archive)


def load_har(archive: Path) -> xr.DataArray:
    """(trial, channel, time) windows with activity, subject, split and segment coordinates."""
    pieces = []
    with zipfile.ZipFile(archive) as z:
        codes = dict(line.split() for line in z.read(f"{ROOT}/activity_labels.txt").decode().splitlines() if line)
        for split in ("train", "test"):
            signals = np.stack(
                [np.loadtxt(z.open(f"{ROOT}/{split}/Inertial Signals/{ch}_{split}.txt")) for ch in CHANNELS], axis=1
            )
            activity = np.loadtxt(z.open(f"{ROOT}/{split}/y_{split}.txt"), dtype=int)
            subject = np.loadtxt(z.open(f"{ROOT}/{split}/subject_{split}.txt"), dtype=int)
            if signals.shape != (len(activity), len(CHANNELS), 128) or len(subject) != len(activity):
                raise ValueError(f"{split}: inconsistent shapes {signals.shape}, {activity.shape}, {subject.shape}")
            pieces.append(
                xr.DataArray(
                    signals,
                    dims=("trial", "channel", "time"),
                    coords={
                        "channel": CHANNELS,
                        "time": np.arange(128) / FS,
                        "activity": ("trial", [codes[str(code)].lower() for code in activity]),
                        "subject": ("trial", subject),
                        "split": ("trial", np.full(len(activity), split)),
                    },
                )
            )
    data = xr.concat(pieces, dim="trial", join="exact")  # fails loudly if channels or times differ
    # A segment is a run of consecutive windows (file order) from one subject doing one activity.
    activity, subject = data["activity"].values, data["subject"].values
    new_segment = np.r_[True, (activity[1:] != activity[:-1]) | (subject[1:] != subject[:-1])]
    return data.assign_coords(trial=np.arange(data.sizes["trial"]), segment=("trial", np.cumsum(new_segment)))

In [3]:
data = load_har(fetch_har(DATA_DIR))
print(dict(data.sizes))

{'trial': 10299, 'channel': 9, 'time': 128}


## The usual starting point: arrays and a scikit-learn model

Six summary statistics and five band powers per channel give 99 features per window. This is the plain NumPy version most projects start from: `X`, `y` and the person each window came from.

In [4]:
def window_features(x: np.ndarray, fs: float) -> np.ndarray:
    """Summary statistics and log band power over the last axis (time); returns (..., len(FEATURES))."""
    power = np.abs(np.fft.rfft(x - x.mean(axis=-1, keepdims=True), axis=-1)) ** 2
    freqs = np.fft.rfftfreq(x.shape[-1], d=1 / fs)
    band_power = [power[..., (freqs >= lo) & (freqs < hi)].sum(axis=-1) for lo, hi in BANDS]
    stats = [x.mean(-1), x.std(-1), x.min(-1), x.max(-1), *np.percentile(x, [25, 75], axis=-1)]
    return np.stack(stats + [np.log(p + 1e-10) for p in band_power], axis=-1)


X = window_features(data.transpose("trial", "channel", "time").values, FS).reshape(data.sizes["trial"], -1)
y = data["activity"].values
subject_ids = data["subject"].values
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
X.shape

(10299, 99)

## 1. Evaluate it in one call

`groups=subject_ids` keeps each person's windows on one side of every split, so the score is for people the model has never seen.

In [5]:
result = xdflow.evaluate(model, X, y, groups=subject_ids)
result

accuracy 0.924 ± 0.029 (mean ± sd over 5 folds, 10299 trials)
Split: 5-fold, grouped (30 groups), stratified by class
Computed once: nothing. Refit in each fold: pipeline (standardscaler, logisticregression).

By group:
       accuracy  n_trials
group                    
1         0.988       347
2         0.977       302
3         0.918       341
4         0.953       317
5         0.911       302
6         0.880       325
...         ...       ...
25        0.885       409
26        0.929       392
27        0.981       376
28        0.911       382
29        0.988       344
30        0.990       383

Split check: nothing shared between training and test sets (group, identical samples).

Every out-of-fold prediction keeps its metadata, so any breakdown is one line:

In [6]:
result.by("group").sort_values("accuracy").head()  # the five hardest people

,accuracy,n_trials
group,,
14,0.662539,323
10,0.826531,294
9,0.847222,288
8,0.871886,281
6,0.880000,325


In [7]:
pd.crosstab(result.predictions.y_true, result.predictions.y_pred, normalize="index").round(2)

y_pred,laying,sitting,standing,walking,walking_downstairs,walking_upstairs
y_true,,,,,,
laying,1.00,0.00,0.00,0.00,0.00,0.00
sitting,0.01,0.93,0.07,0.00,0.00,0.00
standing,0.00,0.05,0.94,0.00,0.00,0.00
walking,0.00,0.00,0.00,0.90,0.05,0.05
walking_downstairs,0.00,0.00,0.00,0.06,0.86,0.08
walking_upstairs,0.00,0.00,0.00,0.04,0.07,0.89


## 2. Check a split you already use

A shuffled, stratified K-fold over windows is a common default. `check_split` reports what it shares between training and test sets, with no model involved:

In [8]:
info = pd.DataFrame({"subject": subject_ids, "segment": data["segment"].values})
xdflow.check_split(X, StratifiedKFold(5, shuffle=True, random_state=0), y=y, metadata=info)

Split check over 5 folds: shared data found.
  - The same 'subject' values appear in training and test sets in 5 of 5 folds: the score measures performance on new trials from a subject seen in training, not on a new subject.
  - The same 'segment' values appear in training and test sets in 5 of 5 folds: the score measures performance on new trials from a segment seen in training, not on a new segment.

                   check  values folds shared        examples
                 subject      30          5/5       1, 10, 11
segment (within subject)     400          5/5 1/1, 1/10, 1/11
       identical samples   10299          0/5                

The same people, and even the same stretches of overlapping windows, are in training and test sets, so that split measures performance on people the model has already seen. For comparison, the same model under that split:

In [9]:
shuffled = xdflow.evaluate(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print(f"shuffled windows: {shuffled.score:.3f}, people kept apart: {result.score:.3f}")

shuffled windows: 0.969, people kept apart: 0.924


## 3. Your preprocessing as a pipeline step

`@xdflow.step` turns a function of the labelled data into a step. It runs once on all windows (each window's features depend on that window alone, which xdflow checks), and the model is refit inside each fold. The data now stays labelled from raw windows to predictions.

In [10]:
@xdflow.step
def features(data: xr.DataArray, fs: float = FS) -> xr.DataArray:
    """Per-channel window features: the time dim becomes a feature dim; every other coordinate is kept."""
    return xr.apply_ufunc(
        window_features, data, input_core_dims=[["time"]], output_core_dims=[["feature"]], kwargs={"fs": fs}
    ).assign_coords(feature=FEATURES)


pipeline_result = xdflow.evaluate(
    [features(), FlattenTransform(dims=("channel", "feature")), model], data, "activity", groups="subject"
)
pipeline_result

accuracy 0.924 ± 0.029 (mean ± sd over 5 folds, 10299 trials)
Split: 5-fold, grouped by subject (30 values), stratified by class
Computed once: features, flatten. Refit in each fold: pipeline (standardscaler, logisticregression).

By subject:
         accuracy  n_trials
subject                    
1           0.988       347
2           0.977       302
3           0.918       341
4           0.953       317
5           0.911       302
6           0.880       325
...           ...       ...
25          0.885       409
26          0.929       392
27          0.981       376
28          0.911       382
29          0.988       344
30          0.990       383

Split check: nothing shared between training and test sets (subject, segment (within subject), identical samples).

## 4. Back into scikit-learn

`to_sklearn` makes the xdflow pipeline a scikit-learn estimator, for `cross_val_score`, `GridSearchCV` or MOABB. With the same folds it gives the same scores:

In [11]:
estimator = xdflow.to_sklearn(
    pipeline_result.pipeline, dims=("trial", "channel", "time"), coords={"channel": CHANNELS, "time": data.time.values}
)
sklearn_scores = cross_val_score(estimator, data.values, y, groups=subject_ids, cv=GroupKFold(5))
xdflow_scores = xdflow.evaluate(pipeline_result.pipeline, data, groups="subject", cv=GroupKFold(5)).fold_scores
np.round(sklearn_scores, 4), np.round(xdflow_scores, 4)

(array([0.9352, 0.9614, 0.9379, 0.9594, 0.8866]),
 array([0.9352, 0.9614, 0.9379, 0.9594, 0.8866]))

## Next

- [Evaluate an Existing Model](https://xdflow.readthedocs.io/en/latest/tutorials/evaluate/): the full guide to `evaluate`, `check_split` and `to_sklearn`.
- [Activity Recognition: Splitting by Subject](https://xdflow.readthedocs.io/en/latest/tutorials/har-activity/): this dataset in depth, with holdout subjects and other models.
- [Writing Custom Transforms](https://xdflow.readthedocs.io/en/latest/guides/writing-transforms/): `@xdflow.step`, including steps that learn from training data.